## Setup the environment

In [ ]:
!pip install faker sodapy --quiet

In [ ]:
dataset = bigquery.Dataset(f"{PROJECT_ID}.{STAGING_BQ_DATASET}")
dataset.location = LOCATION
dataset = bigquery_client.create_dataset(dataset, exists_ok=True)

In [ ]:
from csv import DictWriter
from sodapy import Socrata

FILENAME = "raw-mta-data.csv"
fieldnames = [
    "transit_timestamp",
    "transit_mode",
    "station_complex_id",
    "station_complex",
    "borough",
    "payment_method",
    "fare_class_category",
    "ridership",
    "transfers",
    "latitude",
    "longitude",
    "georeference",
    ":@computed_region_kjdx_g34t",
    ":@computed_region_yamh_8v7k",
    ":@computed_region_wbg7_3whc",
]

# This function will use the api to download the data into a csv locally
# Note that the final size of the CSV would be around 17GB
# also note that the API is slow and prone to timeout errors
# this is why it has a back-off mechanism where we start with the
# maximum records allowed and back off whenever we have a timeout error
# as mentioned above, a much easier approach would be to go to the website
# and download the CSV manually (although that takes some time as well)
# then upload the CSV to GCS
# This method is here for convenience and is not currently being called.

# this method was planned to be able to resume from where it last stopped
# we have the option to read an existing CSV that we started downloading
# if you want to ignore the existing file, you can delete it manually or just flip this flag
FORCE_CLEAR_DATA = False

# I got this number after downloading the full file and looking at it.
# Since this should be a static dataset, the number shouldn't change over time
TOTAL_NUMBER_OF_RECORDS = 110_696_370
STEP = 50_000


def programmatically_download_mta_data():
    import requests

    client = Socrata("data.ny.gov", None)

    # is there is current file already exists
    existing_file = os.path.exists(FILENAME)

    if not existing_file or FORCE_CLEAR_DATA:
        # if we no current file exists, or flag to ignore it is raised
        rows_got = 0
        with open(FILENAME, "w") as mta_fp:
            mta_writer = DictWriter(mta_fp, fieldnames=fieldnames)
            # write headers
            mta_writer.writeheader()
    else:
        with open(FILENAME, "r") as f:
            # read how many records we have already (minus 1 for headers)
            rows_got = sum((1 for _ in f)) - 1
        print(
            f"""Starting from existing data. already got {rows_got:,}
        records ({round(rows_got / TOTAL_NUMBER_OF_RECORDS * 100, 2)}%)"""
        )
    current_step = STEP
    # while the number of rows we got is smaller than the total number of rows expected
    while rows_got < TOTAL_NUMBER_OF_RECORDS:
        try:
            # get more data
            results = client.get("wujg-7c2s", limit=current_step, offset=rows_got)
        except requests.exceptions.ReadTimeout:
            # in case of a timeout, ask for less data
            current_step = current_step - 1000
            print(f"Got timeout, adjusting limit to {current_step}")
        else:
            # when we get data, append it to the file.
            with open(FILENAME, "a") as mta_fp:
                mta_writer = DictWriter(mta_fp, fieldnames=fieldnames)
                mta_writer.writerows(results)
            rows_got = rows_got + len(results)
            print(
                f"Got {rows_got:,} rows so far ({round(rows_got / TOTAL_NUMBER_OF_RECORDS * 100, 2)}%)"
            )

    # TODO: implement code to upload the local CSV to GCS

In [ ]:
# Load raw data to a BQ, to continue transformation of the data
# Due to its size, it will be easier to load data to bq and transform it there

from google.cloud.bigquery import SchemaField

# schema for the original raw file
mta_schema = [
    # Note that the timestamp col is loaded as string, due to US format, which is not automatically detected by BQ
    SchemaField("transit_timestamp", "STRING"),
    SchemaField("transit_mode", "STRING"),
    SchemaField("station_complex_id", "STRING"),
    SchemaField("station_complex", "STRING"),
    SchemaField("borough", "STRING"),
    SchemaField("payment_method", "STRING"),
    SchemaField("fare_class_category", "STRING"),
    SchemaField("ridership", "INTEGER"),
    SchemaField("transfers", "INTEGER"),
    SchemaField("latitude", "FLOAT"),
    SchemaField("longitude", "FLOAT"),
    SchemaField("Georeference", "GEOGRAPHY"),
]

BQ_TABLE = "raw_mta_data"

table_ref = dataset.table(BQ_TABLE)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    schema=mta_schema,
)
load_job = bigquery_client.load_table_from_uri(
    MTA_RAW_CSV, table_ref, job_config=job_config
)
load_job.result()

print("created {}.{}".format(STAGING_BQ_DATASET, BQ_TABLE))

In [ ]:
bigquery_client.query(
    f"DROP TABLE IF EXISTS {STAGING_BQ_DATASET}.mta_data_stations;"
).result()

_query = f"""
CREATE TABLE {STAGING_BQ_DATASET}.mta_data_stations AS
SELECT
  CAST(REPLACE(station_complex_id, 'TRAM', '98765') AS INT64) AS station_id,
  station_complex,
  borough,
  latitude,
  longitude,
FROM
  (
    SELECT
      *,
      ROW_NUMBER() OVER (PARTITION BY station_complex_id ORDER BY transit_timestamp ASC) as rn
    FROM
      `{STAGING_BQ_DATASET}.raw_mta_data`
  )
WHERE
  rn = 1;
"""
bigquery_client.query(_query).result()

stations_df = bigquery_client.query(
    f"SELECT * FROM {STAGING_BQ_DATASET}.mta_data_stations;"
).to_dataframe()
stations_df

In [ ]:
bigquery_client.query(
    f"DROP TABLE IF EXISTS {STAGING_BQ_DATASET}.mta_data_parsed;"
).result()

_query = f"""
    CREATE TABLE `{STAGING_BQ_DATASET}.mta_data_parsed` AS
        SELECT
            PARSE_TIMESTAMP('%m/%d/%Y %I:%M:%S %p', transit_timestamp) AS `transit_timestamp`,
            CAST(REPLACE(station_complex_id, 'TRAM', '98765') AS INT64) AS `station_id`,
            SUM(ridership) as ridership
        FROM `{STAGING_BQ_DATASET}.raw_mta_data`
        GROUP BY PARSE_TIMESTAMP('%m/%d/%Y %I:%M:%S %p', transit_timestamp),
        CAST(REPLACE(station_complex_id, 'TRAM', '98765') AS INT64);
"""
bigquery_client.query(_query).result()
bigquery_client.query(
    f"SELECT * FROM {STAGING_BQ_DATASET}.mta_data_parsed LIMIT 20;"
).to_dataframe()

In [ ]:
bigquery_client.query(f"DROP TABLE IF EXISTS {STAGING_BQ_DATASET}.ridership;").result()
_query = f"""
    CREATE TABLE `{STAGING_BQ_DATASET}.ridership` AS
    SELECT
        TIMESTAMP_ADD(t.transit_timestamp, INTERVAL minute_offset MINUTE) AS transit_timestamp,
        t.station_id,
        CAST(ROUND(
            (FLOOR(t.ridership / 60)) +
            CASE
                WHEN minute_offset < MOD(t.ridership, 60) THEN 1
                ELSE 0
            END
        ) AS INTEGER) AS ridership
    FROM {STAGING_BQ_DATASET}.mta_data_parsed AS t,
    UNNEST(GENERATE_ARRAY(0, 59)) AS minute_offset
    ORDER BY station_id, transit_timestamp;
"""
bigquery_client.query(_query).result()
bigquery_client.query(
    f"SELECT * FROM {STAGING_BQ_DATASET}.ridership LIMIT 20;"
).to_dataframe()

## Generate fake data for bus lines and bus stops (routes)

This is based on the stations data in thr original dataset, but the station addresses are faked (using faker),
and then routes are being constructed just by picking randomly from the stations list. We're using `random.sample`
to select a range but without duplicates (as that might create a circular route for a bus, which is not a typical route)

We will then load the faked data into BigQuery, in order to create time windows of the ridership (that represents people waiting in stations) to simulate bus riders, accumulating riders into a bus driving through their stations.

In [ ]:
with open("bus_stations.csv", "w") as fp:
    writer = DictWriter(fp, fieldnames=fake_stations_lst[0].keys())
    writer.writeheader()
    writer.writerows(fake_stations_lst)

import json

with open("bus_lines.json", "w") as fp:
    for line in bus_lines:
        json.dump(line, fp)
        fp.write("\n")

In [ ]:
bus_lines_schema = [
    SchemaField("bus_line_id", "INTEGER"),
    SchemaField("bus_line", "STRING"),
    SchemaField("number_of_stops", "INTEGER"),
    SchemaField("stops", "INTEGER", mode="REPEATED"),
    SchemaField("frequency_minutes", "INTEGER"),
]

BQ_TABLE = "bus_lines"

dataset = bigquery.Dataset(f"{PROJECT_ID}.{STAGING_BQ_DATASET}")


table_ref = dataset.table(BQ_TABLE)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    schema=bus_lines_schema,
)
with open("bus_lines.json", "rb") as fp:
    load_job = bigquery_client.load_table_from_file(
        fp, table_ref, job_config=job_config
    )
    load_job.result()

print("created {}.{}".format(STAGING_BQ_DATASET, BQ_TABLE))

## Extract data to GCS

Now, that we have all data we need, the data needs to reside in BigQuery & GCS, we will extract the BigQuery tables to GCS, so we can load them in the next notebook, where we build the open source data lakehouse.

- `bus_stations` - we will just use our CSV we built locally, and upload it as is to GCS.

- `bus_lines` - we will export the data as `parquet` format, to be read by spark in the next notebook, to demonstrate iceberg **external** tables.

- `ridership` - this table is the largest by far (972,829,500 rows). We will export this table to parquet format to be loaded as a managed iceberg table.

In [ ]:
from google.cloud.bigquery import ExtractJobConfig

target_glob = "mta_staging_data/bus_lines"
destination_uri = "gs://{}/{}/*.parquet".format(BUCKET_NAME, target_glob)

blob_list = bucket.list_blobs(prefix=target_glob)
blob_list = [blob for blob in blob_list]
bucket.delete_blobs(blob_list)

job_config = ExtractJobConfig()
job_config.destination_format = bigquery.DestinationFormat.PARQUET

table_ref_1 = dataset.table("bus_lines")
extract_job = bigquery_client.extract_table(
    table_ref_1, destination_uri, location=LOCATION, job_config=job_config
)
extract_job.result()

In [ ]:
try:
    for table in bigquery_client.list_tables(dataset):
        bigquery_client.delete_table(table)
    bigquery_client.delete_dataset(dataset)
except exceptions.NotFound:
    print("Dataset looks already dropped")